# RoboBarista – unser gewählter Standardaufbau

**FAIRINO FR5 · 3 virtuelle RGB-D-Kameras · 1 seitlicher AprilTag 36h11 am Milchkännchen · AprilTag 3 (Pupil Labs) · gemeinsame 6D-Optimierung.**

Wir haben am 21.10.2026 bereits 16 Alternativen in Colab verglichen. Unsere gewählte einfache Variante erreichte in **drei statischen Simulationsposen** höchstens **0,138 mm absolute Abweichung der Ausgießerspitze** und **0,046° Winkelabweichung**; beide Versuchs-Gates waren PASS. **Das ist kein Nachweis für dynamische Bewegungen oder echte Sensoren.**

In diesem Notebook testen wir nur noch die **gewählte Standardkonfiguration**. Der aufwendigere Vergleich von 16 Varianten ist weiter in [COLAB_COMPARE.ipynb](https://colab.research.google.com/github/Coffee-Tobi/robocasa/blob/main/COLAB_COMPARE.ipynb) verfügbar und wurde nicht gelöscht.

**Start:** Hardwarebeschleuniger »Keine« genügt. Danach **Laufzeit → Alle ausführen**.


In [ ]:
# 1. GitHub-Projekt laden (keine ZIP nötig)
import os, subprocess, sys, pathlib
repo = pathlib.Path('/content/robobarista-vision')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Coffee-Tobi/robocasa.git', str(repo)], check=True)
else:
    # Bei erneutem Ausführen aktuelle Verbesserungen von GitHub holen.
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
os.chdir(repo)
print('Projekt:', repo)


In [ ]:
# 2. Systemgrafik (CPU/OSMesa) und Python-Pakete installieren
# Ohne GPU benötigt MuJoCo echte OSMesa-Systembibliotheken; pip allein reicht nicht.
import os, subprocess, sys, ctypes.util
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'libosmesa6-dev',
                'libgl1', 'libglx-mesa0', 'libgl1-mesa-dri'], check=True)
osmesa_lib = ctypes.util.find_library('OSMesa')
print('OSMesa-Systembibliothek:', osmesa_lib)
if not osmesa_lib:
    raise RuntimeError('OSMesa fehlt trotz apt-Installation. Bitte die Ausgabe von Schritt 2 teilen.')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', 'requirements.txt'], check=True)

# Eigenständiger Importcheck im richtigen Rendering-Backend (ohne GPU: OSMesa).
backend = 'egl' if os.path.exists('/dev/nvidia0') else 'osmesa'
check_env = os.environ.copy()
check_env['MUJOCO_GL'] = backend
if backend == 'osmesa':
    check_env['PYOPENGL_PLATFORM'] = 'osmesa'
print('MuJoCo-Importtest mit Backend:', backend, flush=True)
subprocess.run([sys.executable, '-c',
                'import mujoco; print("MuJoCo:", mujoco.__version__, "Import OK")'],
               env=check_env, check=True)
import cv2, numpy as np
print('OpenCV:', cv2.__version__, 'NumPy:', np.__version__)


In [ ]:
# 3. Mathematische Tests – vollständige Fehleranzeige statt kryptischem Exit-Code
import subprocess,sys
test = subprocess.run(
    [sys.executable, '-m', 'pytest', '-vv', '-ra', '--tb=short', 'tests'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    text=True, errors='replace', check=False)
print('=== Testergebnisse (stdout) ===', flush=True)
print(test.stdout[-30000:] if test.stdout.strip() else '(leer)', flush=True)
print('=== Zusätzliche Fehlermeldungen (stderr) ===', flush=True)
print(test.stderr[-20000:] if test.stderr.strip() else '(leer)', flush=True)
if test.returncode:
    raise RuntimeError(
        'Mindestens ein Python-Test ist fehlgeschlagen. '
        'Oben steht der genaue Testname unter FAILURES/ERRORS; '
        'bitte diesen Abschnitt teilen. Nicht mit unvalidierten Messungen fortfahren.')
print('Alle Python-Tests bestanden.')


In [ ]:
# 4. Standardaufbau: drei Kameras + ein seitlicher AprilTag 3
# Entsprechend dem am 21.10.2026 gewaehlten und archivierten Benchmark.
import os, subprocess, sys
settings=os.environ.copy()
settings['MUJOCO_GL']='egl' if os.path.exists('/dev/nvidia0') else 'osmesa'
if settings['MUJOCO_GL']=='osmesa':
    settings['PYOPENGL_PLATFORM']='osmesa'
cmd=[sys.executable,'-u','-m','robobarista.demo',
     '--pitcher-side-tags','--pitcher-tags','1','--cameras','3',
     '--detector','pupil','--pose-method','bundle_adjust',
     '--trials','3','--width','1280','--height','960',
     '--output','runs/mvp_pitcher','--min-cameras','2',
     '--max-spout-mm','0.5','--max-pitcher-rotation-deg','1.0']
print('Rendering:',settings['MUJOCO_GL'],'| rig: 3 Kameras / 1 AprilTag 3',flush=True)
run=subprocess.run(cmd,cwd=str(repo),env=settings,
                   stdout=subprocess.PIPE,stderr=subprocess.PIPE,
                   text=True,errors='replace',check=False)
print('Benchmark Exit-Code:',run.returncode)
print('\n=== PROGRAMMAUSGABE ===')
print(run.stdout[-18000:] if run.stdout.strip() else '(leer)')
print('\n=== FEHLERHINWEISE ===')
print(run.stderr[-14000:] if run.stderr.strip() else '(keine)')
if run.returncode:
    print('Nicht bestanden: Bitte Messbericht und Fehlermeldungen kontrollieren.')
else:
    print('Standardaufbau hat die aktuelle statische Simulation bestanden.')


In [ ]:
# 5. Kameras und Tiefenbilder nebeneinander ansehen
from IPython.display import display,Image
from pathlib import Path
out=Path(repo)/'runs/mvp_pitcher'
for filename in ['rgbd_montage.jpg','pose_comparison.png']:
    f=out/filename
    if f.exists():
        print(filename)
        display(Image(filename=str(f)))
    else:
        print('Nicht vorhanden:',f)


In [ ]:
# 6. Genaue Abweichungen ansehen und Bilder/CSV bei Bedarf herunterladen
import json,pandas as pd
p=out/'summary.json'
if p.exists():
    report=json.loads(p.read_text())
    print(json.dumps(report,indent=2,ensure_ascii=False))
    print('Kännchenprüfung:',report.get('pitcher_status'))
    print('Gesamtszene:',report.get('status'))
    print('Spitzenfehler (max.):',report.get('max_spout_tip_error_mm'),'mm')
    error_csv=out/'pose_errors.csv'
    if error_csv.exists(): display(pd.read_csv(error_csv))
else:
    print('Kein Messbericht. Bitte Fehlermeldung aus Zelle 4 schicken.')


### Die bewegte Demonstration ist jetzt verfügbar! 🎬

Die bisherige Kamera-Baseline bleibt in diesem Notebook erhalten. **Der nächste Schritt ist bereits in einem eigenen Notebook umgesetzt:**

**[COLAB_MOTION.ipynb – Milchkännchen bewegt sich, drei Kameras zeichnen auf](https://colab.research.google.com/github/Coffee-Tobi/robocasa/blob/main/COLAB_MOTION.ipynb)**

Dort siehst du die animierte virtuelle Gießgeste und erhältst die zeitgestempelten, ausschließlich aus Kamerabildern geschätzten Kännchen- und Ausgießerpositionen. Es handelt sich noch um eine synthetische Bewegungsfolge ohne physisches Milchgießen und ohne FR5-Gelenkausführung.

[Projektplan](https://github.com/Coffee-Tobi/robocasa/blob/main/planning/ROBOBARISTA_DREI_STUFEN_MVP.md)
